## Thực hành 3: Làm quen với cơ sở dữ liệu SQL

In [8]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

In [9]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Đọc dữ liệu từ file CSV, thay đường dẫn file của anh chị vào 
df = pd.read_csv(r"D:\NguyenMinhHuong\TA\SS_Bigdata\data\ai4i2020.csv")

# 2. Chuẩn hóa tên cột sang chữ thường và thay ký tự đặc biệt bằng dấu gạch dưới (snake_case)
df.columns = [
    c.lower()
     .replace(" [k]", "_k")
     .replace(" [rpm]", "_rpm")
     .replace(" [nm]", "_nm")
     .replace(" [min]", "_min")
     .replace(" ", "_")
    for c in df.columns
]

print("Danh sách các cột sau khi chuẩn hóa:")
print(df.columns.tolist())

# 3. Cấu hình thông tin kết nối PostgreSQL từ file docker-compose.yml
DB_USER = "bigdata_user"
DB_PASS = "BigDataPass123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "bigdata_db"

# 4. Tạo Engine kết nối SQLAlchemy
connection_str = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_str)

# 5. Nạp dữ liệu vào bảng 'ai4i2020' (nếu bảng đã có sẽ ghi đè 'replace')
table_name = "ai4i2020"
df.to_sql(name=table_name, con=engine, if_exists="replace", index=False)

print(f"\n Đã nạp thành công {len(df)} dòng dữ liệu vào bảng '{table_name}' trong PostgreSQL!")

Danh sách các cột sau khi chuẩn hóa:
['udi', 'product_id', 'type', 'air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min', 'machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf']

 Đã nạp thành công 10000 dòng dữ liệu vào bảng 'ai4i2020' trong PostgreSQL!


### Hướng dẫn nhanh về các câu lệnh SQL cơ bản & nâng cao
- Hãy tưởng tượng tập dữ liệu của bạn như một bảng Excel lớn trong Database, SQL chính là bộ công cụ giúp bạn lọc, gom nhóm, nối bảng và tính toán trên bảng đó.
------------------------------
### 1. SELECT & FROM — Chọn cột và Nguồn dữ liệu
* **Ý nghĩa:** `FROM` xác định bảng nguồn. `SELECT` quyết định những cột nào cần hiển thị.
* **Ví dụ:** `SELECT product_id, type FROM ai4i2020 LIMIT 10;`

### 2. WHERE — Bộ lọc dòng (Filter)
* **Ý nghĩa:** Lọc các dòng thỏa mãn điều kiện logic (như `machine_failure = 1`).

### 3. ORDER BY — Sắp xếp dữ liệu (Sort)
* **Ý nghĩa:** Sắp xếp tăng dần (`ASC` - mặc định) hoặc giảm dần (`DESC`).

### 4. GROUP BY & AGGREGATION — Gom cụm và Hàm tổng hợp
* **Ý nghĩa:** Gom nhóm các giá trị trùng nhau (như Pivot Table). Đi kèm các hàm: `COUNT(*)`, `SUM()`, `AVG()`, `MAX()`, `MIN()`.

### 5. HAVING — Bộ lọc sau khi gom cụm
* **Ý nghĩa:** Lọc kết quả của các nhóm sau khi chạy `GROUP BY` (khác với `WHERE` là lọc từng dòng thô trước khi gom).

### 6. INNER JOIN — Liên kết hai bảng
* **Ý nghĩa:** Gộp dữ liệu từ 2 bảng khác nhau thông qua khóa liên kết (`Product_ID`).

### 7. CASE WHEN — Câu lệnh điều kiện (If-Else trong SQL)
* **Ý nghĩa:** Phân loại hoặc gán nhãn mới cho dữ liệu trực tiếp trong câu truy vấn.

------------------------------
### ⏳ Thứ tự thực thi của câu lệnh SQL trong Database Engine:
1. **FROM & JOIN** (Xác định nguồn và nối bảng)
2. **WHERE** (Lọc các dòng thô)
3. **GROUP BY** (Gom nhóm)
4. **HAVING** (Lọc các nhóm)
5. **SELECT** (Trích xuất cột & tính hàm tổng hợp)
6. **ORDER BY** (Sắp xếp kết quả)
7. **LIMIT** (Giới hạn số dòng hiển thị)

###  0. Kiểm tra tên cột và kiểu dữ liệu thực tế trong Database
Trước khi viết các câu lệnh SQL nâng cao, ta truy vấn bảng siêu dữ liệu `information_schema.columns` để nắm chính xác tên và kiểu dữ liệu của các cột trong bảng `ai4i2020`.

In [10]:
# Truy vấn thông tin cấu trúc cột từ information_schema
query_schema = """
SELECT 
    column_name, 
    data_type 
FROM information_schema.columns 
WHERE table_name = 'ai4i2020'
ORDER BY ordinal_position;
"""

df_schema = pd.read_sql(query_schema, con=engine)
display(df_schema)

,column_name,data_type
0,udi,bigint
1,product_id,text
2,type,text
3,air_temperature_k,double precision
4,process_temperature_k,double precision
5,rotational_speed_rpm,bigint
6,torque_nm,double precision
7,tool_wear_min,bigint
8,machine_failure,bigint
9,twf,bigint


### 1. Lọc dữ liệu (WHERE) kết hợp Sắp xếp (ORDER BY)
**Bài toán:** Tìm Top 10 máy có nhiệt độ quy trình (`process_temperature_k`) cao nhất trong số các máy bị lỗi (`machine_failure = 1`) để kiểm tra xem nhiệt độ có phải là nguyên nhân chính gây lỗi hay không.

In [11]:
query_task1 = """
SELECT 
    udi,
    product_id, 
    type, 
    air_temperature_k, 
    process_temperature_k, 
    rotational_speed_rpm, 
    torque_nm, 
    machine_failure
FROM ai4i2020
WHERE machine_failure = 1
ORDER BY process_temperature_k DESC
LIMIT 10;
"""

df_task1 = pd.read_sql(query_task1, con=engine)
print("--- TOP 10 MÁY HỎNG CÓ NHIỆT ĐỘ QUY TRÌNH CAO NHẤT ---")
display(df_task1)

--- TOP 10 MÁY HỎNG CÓ NHIỆT ĐỘ QUY TRÌNH CAO NHẤT ---


,udi,product_id,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,machine_failure
0,5142,L52321,L,304.4,313.7,1509,35.0,1
1,5310,M20169,M,303.9,313.2,1422,48.0,1
2,5049,H34462,H,304.0,313.2,1271,68.6,1
3,4989,L52168,L,303.8,313.1,2497,13.0,1
4,4985,L52164,L,303.8,313.1,1256,58.7,1
5,5220,L52399,L,303.8,313.0,1365,59.9,1
6,5062,L52241,L,304.0,312.9,1363,62.5,1
7,4998,M19857,M,303.6,312.8,2659,11.4,1
8,4977,L52156,L,303.7,312.7,1359,56.8,1
9,5335,M20194,M,303.4,312.6,2706,9.8,1


### 2. Gom cụm (GROUP BY) & Hàm tổng hợp (AGGREGATION)
**Bài toán:** Thống kê tổng số máy (`COUNT(*)`), số lượng máy lỗi (`SUM(machine_failure)`), tỷ lệ lỗi (`AVG(machine_failure) * 100`), nhiệt độ quy trình trung bình (`AVG(process_temperature_k)`) và thời gian hao mòn công cụ lớn nhất (`MAX(tool_wear_min)`) theo từng loại máy (`type` - gồm H, M, L).

In [12]:
query_task2 = """
SELECT 
    type,
    COUNT(*) AS total_machines,
    SUM(machine_failure) AS failed_machines,
    ROUND((AVG(machine_failure) * 100)::numeric, 2) AS failure_rate_pct,
    ROUND(AVG(process_temperature_k)::numeric, 2) AS avg_process_temp_k,
    MAX(tool_wear_min) AS max_tool_wear_min
FROM ai4i2020
GROUP BY type
ORDER BY total_machines DESC;
"""

df_task2 = pd.read_sql(query_task2, con=engine)
print("--- THỐNG KÊ VẬN HÀNH THEO TỪNG LOẠI MÁY (TYPE) ---")
display(df_task2)

--- THỐNG KÊ VẬN HÀNH THEO TỪNG LOẠI MÁY (TYPE) ---


,type,total_machines,failed_machines,failure_rate_pct,avg_process_temp_k,max_tool_wear_min
0,L,6000,235.0,3.92,310.01,251
1,M,2997,83.0,2.77,310.02,253
2,H,1003,21.0,2.09,309.93,246


### 3. Điều kiện sau gom cụm (HAVING)
**Bài toán:** Tìm các nhóm máy (`type`) mà trong đó, tổng số phút hao mòn công cụ của những máy đã bị lỗi vượt quá **500 phút** (giúp xác định dòng máy nào dễ hỏng do hao mòn dao nhất).

In [13]:
query_task3 = """
SELECT 
    type,
    COUNT(*) AS failed_machine_count,
    SUM(tool_wear_min) AS total_wear_failed_min,
    ROUND(AVG(tool_wear_min)::numeric, 2) AS avg_wear_failed_min
FROM ai4i2020
WHERE machine_failure = 1
GROUP BY type
HAVING SUM(tool_wear_min) > 500
ORDER BY total_wear_failed_min DESC;
"""

df_task3 = pd.read_sql(query_task3, con=engine)
print("--- CÁC LOẠI MÁY CÓ TỔNG THỜI GIAN MÒN DAO (KHI LỖI) > 500 PHÚT ---")
display(df_task3)

--- CÁC LOẠI MÁY CÓ TỔNG THỜI GIAN MÒN DAO (KHI LỖI) > 500 PHÚT ---


,type,failed_machine_count,total_wear_failed_min,avg_wear_failed_min
0,L,235,34989.0,148.89
1,M,83,10731.0,129.29
2,H,21,3022.0,143.90


### 4. Liên kết bảng (JOIN)
Trong thực tế, thông tin vận hành cần kết hợp với thông tin vị trí lắp đặt và người phụ trách.
* Giả định có thêm một bảng danh mục là `factory_floors` chứa thông tin phân xưởng và quản lý của từng máy.
* Dùng `INNER JOIN` để kết hợp dữ liệu bảng `ai4i2020` và `factory_floors` thông qua khóa `product_id`.

In [18]:
# 1. Tạo bảng danh mục 'factory_floors' và nạp vào PostgreSQL
df_floors = pd.DataFrame({
    "product_id": df["product_id"].unique(),
    "floor_name": [
        "Phân xưởng A (Dây chuyền 1)" if pid.startswith("L") 
        else ("Phân xưởng B (Dây chuyền 2)" if pid.startswith("M") 
        else "Phân xưởng C (Chất lượng cao)") 
        for pid in df["product_id"].unique()
    ],
    "manager": [
        "Nguyễn Văn A" if pid.startswith("L") 
        else ("Trần Thị B" if pid.startswith("M") 
        else "Lê Văn C") 
        for pid in df["product_id"].unique()
    ]
})

df_floors.to_sql("factory_floors", con=engine, if_exists="replace", index=False)
print("Đã tạo bảng danh mục 'factory_floors' thành công!")

# 2. Truy vấn liên kết bảng với INNER JOIN
query_task4 = """
SELECT 
    m.udi,
    m.product_id,
    m.type,
    f.floor_name,
    f.manager,
    m.process_temperature_k,
    m.rotational_speed_rpm,
    m.torque_nm,
    m.machine_failure
FROM ai4i2020 m
INNER JOIN factory_floors f ON m.product_id = f.product_id
WHERE m.machine_failure = 1
ORDER BY m.udi ASC
LIMIT 10;
"""

df_task4 = pd.read_sql(query_task4, con=engine)
print("\n--- BÁO CÁO MÁY HỎNG KÈM THÔNG TIN PHÂN XƯỞNG VÀ QUẢN LÝ ---")
display(df_task4)

Đã tạo bảng danh mục 'factory_floors' thành công!

--- BÁO CÁO MÁY HỎNG KÈM THÔNG TIN PHÂN XƯỞNG VÀ QUẢN LÝ ---


,udi,product_id,type,floor_name,manager,process_temperature_k,rotational_speed_rpm,torque_nm,machine_failure
0,51,L47230,L,Phân xưởng A (Dây chuyền 1),Nguyễn Văn A,309.1,2861,4.6,1
1,70,L47249,L,Phân xưởng A (Dây chuyền 1),Nguyễn Văn A,309.0,1410,65.7,1
2,78,L47257,L,Phân xưởng A (Dây chuyền 1),Nguyễn Văn A,308.9,1455,41.3,1
3,161,L47340,L,Phân xưởng A (Dây chuyền 1),Nguyễn Văn A,308.2,1282,60.7,1
4,162,L47341,L,Phân xưởng A (Dây chuyền 1),Nguyễn Văn A,308.1,1412,52.3,1
5,169,L47348,L,Phân xưởng A (Dây chuyền 1),Nguyễn Văn A,308.3,1433,62.3,1
6,195,M15054,M,Phân xưởng B (Dây chuyền 2),Trần Thị B,308.5,2678,10.7,1
7,208,M15067,M,Phân xưởng B (Dây chuyền 2),Trần Thị B,308.7,1421,60.7,1
8,243,L47422,L,Phân xưởng A (Dây chuyền 1),Nguyễn Văn A,308.2,1348,58.8,1
9,249,L47428,L,Phân xưởng A (Dây chuyền 1),Nguyễn Văn A,308.3,1362,56.8,1


### 5. Kỹ thuật nâng cao: Phân loại lỗi (CASE WHEN)
Tập dữ liệu chứa các cột nguyên nhân sự cố: `twf` (Hao mòn dao), `hdf` (Tản nhiệt kém), `pwf` (Lỗi công suất), `osf` (Quá tải), `rnf` (Ngẫu nhiên).
* Dưới đây là cách dùng `CASE WHEN` kết hợp hàm `SUM()` để đếm xem từng loại máy thường gặp lỗi gì nhất.
* Đồng thời dùng `CASE WHEN` để gắn nhãn mức độ cảnh báo nhiệt độ của cảm biến.

In [15]:
# 1. Thống kê số lượng từng loại lỗi theo Type bằng CASE WHEN
query_task5 = """
SELECT 
    type,
    COUNT(*) AS total_machines,
    SUM(CASE WHEN twf = 1 THEN 1 ELSE 0 END) AS twf_wear_failures,
    SUM(CASE WHEN hdf = 1 THEN 1 ELSE 0 END) AS hdf_heat_failures,
    SUM(CASE WHEN pwf = 1 THEN 1 ELSE 0 END) AS pwf_power_failures,
    SUM(CASE WHEN osf = 1 THEN 1 ELSE 0 END) AS osf_overstrain_failures,
    SUM(CASE WHEN rnf = 1 THEN 1 ELSE 0 END) AS rnf_random_failures,
    SUM(machine_failure) AS total_failures
FROM ai4i2020
GROUP BY type
ORDER BY total_failures DESC;
"""

df_task5 = pd.read_sql(query_task5, con=engine)
print("--- THỐNG KÊ 5 NGUYÊN NHÂN LỖI THEO TỪNG LOẠI MÁY ---")
display(df_task5)

--- THỐNG KÊ 5 NGUYÊN NHÂN LỖI THEO TỪNG LOẠI MÁY ---


,type,total_machines,twf_wear_failures,hdf_heat_failures,pwf_power_failures,osf_overstrain_failures,rnf_random_failures,total_failures
0,L,6000,25,76,59,87,13,235.0
1,M,2997,14,31,31,9,2,83.0
2,H,1003,7,8,5,2,4,21.0


In [19]:
# 2. Phân loại mức độ rủi ro nhiệt độ bằng CASE WHEN
query_risk = """
SELECT 
    product_id,
    type,
    air_temperature_k,
    process_temperature_k,
    CASE 
        WHEN air_temperature_k >= 302.0 THEN 'Rất nóng (Nguy cơ cao)'
        WHEN air_temperature_k >= 300.0 THEN 'Cảnh báo quá nhiệt'
        ELSE 'Bình thường'
    END AS temp_risk_level,
    machine_failure
FROM ai4i2020
WHERE machine_failure = 1
ORDER BY air_temperature_k DESC
LIMIT 10;
"""

df_risk = pd.read_sql(query_risk, con=engine)
print("--- PHÂN LOẠI MỨC ĐỘ RỦI RO CẢM BIẾN NHIỆT ĐỘ (TOP 10 MÁY HỎNG) ---")
display(df_risk)

--- PHÂN LOẠI MỨC ĐỘ RỦI RO CẢM BIẾN NHIỆT ĐỘ (TOP 10 MÁY HỎNG) ---


,product_id,type,air_temperature_k,process_temperature_k,temp_risk_level,machine_failure
0,L52321,L,304.4,313.7,Rất nóng (Nguy cơ cao),1
1,H34462,H,304.0,313.2,Rất nóng (Nguy cơ cao),1
2,L52241,L,304.0,312.9,Rất nóng (Nguy cơ cao),1
3,M20169,M,303.9,313.2,Rất nóng (Nguy cơ cao),1
4,L52164,L,303.8,313.1,Rất nóng (Nguy cơ cao),1
5,L52168,L,303.8,313.1,Rất nóng (Nguy cơ cao),1
6,L52399,L,303.8,313.0,Rất nóng (Nguy cơ cao),1
7,L52031,L,303.7,312.1,Rất nóng (Nguy cơ cao),1
8,M19622,M,303.7,312.0,Rất nóng (Nguy cơ cao),1
9,H34175,H,303.7,311.9,Rất nóng (Nguy cơ cao),1



### BÀI TẬP VỀ NHÀ: THỰC HÀNH TRUY VẤN SQL CƠ BẢN & NÂNG CAO

**Mục tiêu:** Củng cố kiến thức và rèn luyện kỹ năng viết câu lệnh truy vấn SQL từ cơ bản đến nâng cao trên cơ sở dữ liệu `ai4i2020` và bảng liên kết `factory_floors`:
* **Cơ bản:** `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`
* **Thống kê:** Các hàm tổng hợp (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`)
* **Gom nhóm & Lọc nhóm:** `GROUP BY`, `HAVING`
* **Nối bảng:** `INNER JOIN`
* **Logic điều kiện:** `CASE WHEN`

> **Hướng dẫn làm bài:** 
> * Viết câu lệnh SQL vào biến chuỗi `query_hw...` trong mỗi cell code.
> * Chạy cell để thực thi truy vấn qua `pd.read_sql()` và quan sát bảng kết quả.


#### Bài tập 1: Lọc dữ liệu máy chạy tốc độ cao và hoạt động ổn định (SELECT, WHERE, ORDER BY, LIMIT)
**Bài toán:** Trưởng ca vận hành muốn kiểm tra danh sách các máy loại chất lượng cao đang hoạt động ở dải tốc độ cao nhưng vẫn giữ được độ ổn định (chưa từng bị hỏng).

**Yêu cầu:**
1. Lấy dữ liệu từ bảng `ai4i2020`.
2. Điều kiện lọc (`WHERE`):
   - Thuộc dòng máy chất lượng cao (`type = 'H'`).
   - Tốc độ quay (`rotational_speed_rpm`) đạt từ **1500 vòng/phút trở lên** (`>= 1500`).
   - Máy hoạt động bình thường, **không bị hỏng** (`machine_failure = 0`).
3. Cột cần lấy: `product_id`, `type`, `rotational_speed_rpm`, `torque_nm`, `tool_wear_min`.
4. Sắp xếp (`ORDER BY`): Tốc độ quay (`rotational_speed_rpm`) giảm dần (`DESC`).
5. Giới hạn (`LIMIT`): Lấy đúng **10 bản ghi đầu tiên**.


In [ ]:
# --- BÀI TẬP 1: Viết câu truy vấn SQL của bạn tại đây ---
query_hw1 = """
SELECT 
    -- Điền danh sách các cột cần lấy
FROM ai4i2020
WHERE 
    -- Điền các điều kiện lọc kết hợp với AND
ORDER BY 
    -- Điền cột cần sắp xếp và hướng sắp xếp
LIMIT 10;
"""

df_hw1 = pd.read_sql(query_hw1, con=engine)
print(f"Kết quả Bài 1 (Số dòng lấy ra: {len(df_hw1)}):")
display(df_hw1)


#### Bài tập 2: Thống kê tổng quan các chỉ số vận hành toàn nhà máy (AGGREGATION - COUNT, AVG, MIN, MAX)
**Bài toán:** Bộ phận phân tích dữ liệu cần một bảng báo cáo tóm tắt các chỉ số thống kê cơ bản trên toàn bộ 10,000 dòng dữ liệu của nhà máy.

**Yêu cầu:**
Viết câu truy vấn tính toán các giá trị sau từ bảng `ai4i2020`:
1. Tổng số máy được theo dõi (`COUNT(*)`), đặt alias là `total_records`.
2. Nhiệt độ quy trình trung bình (`AVG(process_temperature_k)`), làm tròn 2 chữ số thập phân, đặt alias là `avg_process_temp`.
3. Momen xoắn nhỏ nhất (`MIN(torque_nm)`), đặt alias là `min_torque`.
4. Momen xoắn lớn nhất (`MAX(torque_nm)`), đặt alias là `max_torque`.
5. Thời gian mòn dao trung bình (`AVG(tool_wear_min)`), làm tròn 2 chữ số thập phân, đặt alias là `avg_tool_wear`.

*(Gợi ý: Trong PostgreSQL, sử dụng `ROUND(AVG(...)::numeric, 2)` để làm tròn).*


In [ ]:
# --- BÀI TẬP 2: Viết câu truy vấn SQL của bạn tại đây ---
query_hw2 = """
SELECT 
    -- Sử dụng các hàm tổng hợp COUNT, AVG, MIN, MAX và đặt alias tương ứng
FROM ai4i2020;
"""

df_hw2 = pd.read_sql(query_hw2, con=engine)
print("Kết quả Bài 2 (Tổng quan chỉ số vận hành toàn nhà máy):")
display(df_hw2)


#### Bài tập 3: So sánh đặc trưng vận hành giữa máy hỏng và máy bình thường (GROUP BY & AGGREGATION)
**Bài toán:** Để tìm hiểu các thông số nào có sự chênh lệch lớn giữa trạng thái bình thường và trạng thái lỗi, hãy gom nhóm dữ liệu theo cột `machine_failure`.

**Yêu cầu:**
Gom nhóm theo trạng thái lỗi của máy (`machine_failure` gồm `0` và `1`) và tính:
1. Cột trạng thái: `machine_failure`.
2. Số lượng máy trong từng nhóm (`COUNT(*)`), đặt alias là `machine_count`.
3. Nhiệt độ quy trình trung bình (`ROUND(AVG(process_temperature_k)::numeric, 2)`), đặt alias là `avg_process_temp`.
4. Momen xoắn trung bình (`ROUND(AVG(torque_nm)::numeric, 2)`), đặt alias là `avg_torque`.
5. Tốc độ quay trung bình (`ROUND(AVG(rotational_speed_rpm)::numeric, 2)`), đặt alias là `avg_speed`.
6. Thời gian mòn dao trung bình (`ROUND(AVG(tool_wear_min)::numeric, 2)`), đặt alias là `avg_tool_wear`.


In [ ]:
# --- BÀI TẬP 3: Viết câu truy vấn SQL của bạn tại đây ---
query_hw3 = """
SELECT 
    machine_failure,
    -- Điền các hàm tổng hợp tương ứng
FROM ai4i2020
GROUP BY machine_failure;
"""

df_hw3 = pd.read_sql(query_hw3, con=engine)
print("Kết quả Bài 3 (So sánh nhóm máy bình thường vs máy hỏng):")
display(df_hw3)


#### Bài tập 4: Lọc nhóm máy có tần suất hỏng do mòn dao đáng báo động (GROUP BY & HAVING)
**Bài toán:** Đội bảo trì muốn xác định những dòng máy (`type`) có số lượng ca hỏng hóc do mòn dao (`twf`) ở mức cao để ưu tiên thay phụ tùng định kỳ.

**Yêu cầu:**
Gom nhóm theo loại máy (`type`):
1. Cột phân loại: `type`.
2. Tổng số ca hỏng do mòn dao (`SUM(twf)`), đặt alias là `twf_failure_count`.
3. Tổng số phút mòn dao của cả nhóm (`SUM(tool_wear_min)`), đặt alias là `total_tool_wear`.
4. Thời gian mòn dao trung bình (`ROUND(AVG(tool_wear_min)::numeric, 2)`), đặt alias là `avg_tool_wear`.
5. **Điều kiện lọc nhóm (HAVING):** Chỉ giữ lại các nhóm máy có tổng số ca hỏng do mòn dao từ 10 máy trở lên (`SUM(twf) >= 10`).
6. **Sắp xếp (ORDER BY):** Tổng thời gian mòn dao (`total_tool_wear`) giảm dần (`DESC`).


In [ ]:
# --- BÀI TẬP 4: Viết câu truy vấn SQL của bạn tại đây ---
query_hw4 = """
SELECT 
    type,
    -- Điền các hàm tính toán tổng hợp
FROM ai4i2020
GROUP BY type
HAVING 
    -- Điền điều kiện lọc sau gom nhóm (SUM(twf) >= 10)
ORDER BY 
    -- Điền điều kiện sắp xếp giảm dần;
"""

df_hw4 = pd.read_sql(query_hw4, con=engine)
print("Kết quả Bài 4 (Các dòng máy có nguy cơ mòn dao cao):")
display(df_hw4)


#### Bài tập 5: Báo cáo danh sách sự cố nhiệt và quá tải kèm thông tin phân xưởng (INNER JOIN & WHERE)
**Bài toán:** Trưởng phòng kỹ thuật cần danh sách chi tiết các máy gặp sự cố do tản nhiệt kém (`hdf = 1`) hoặc quá tải (`osf = 1`), kèm theo thông tin phân xưởng và người quản lý chịu trách nhiệm.

**Yêu cầu:**
Kết hợp bảng `ai4i2020` (đặt alias là `m`) và bảng `factory_floors` (đặt alias là `f`) qua khóa chung `product_id`:
1. Điều kiện lọc (`WHERE`): Máy gặp sự cố tản nhiệt kém (`m.hdf = 1`) **HOẶC** sự cố quá tải (`m.osf = 1`).
2. Cột cần lấy: `m.product_id`, `m.type`, `f.floor_name`, `f.manager`, `m.torque_nm`, `m.process_temperature_k`, `m.hdf`, `m.osf`.
3. Sắp xếp (`ORDER BY`): Theo tên phân xưởng `f.floor_name` tăng dần (`ASC`), sau đó theo momen xoắn `m.torque_nm` giảm dần (`DESC`).
4. Giới hạn (`LIMIT`): Lấy đúng **15 bản ghi đầu tiên**.


In [ ]:
# --- BÀI TẬP 5: Viết câu truy vấn SQL của bạn tại đây ---
query_hw5 = """
SELECT 
    -- Điền các cột cần lấy từ m và f
FROM ai4i2020 m
INNER JOIN factory_floors f ON m.product_id = f.product_id
WHERE 
    -- Điều kiện lọc: m.hdf = 1 OR m.osf = 1
ORDER BY 
    -- Sắp xếp theo phân xưởng ASC, sau đó theo momen xoắn DESC
LIMIT 15;
"""

df_hw5 = pd.read_sql(query_hw5, con=engine)
print(f"Kết quả Bài 5 (Danh sách 15 sự cố nhiệt / quá tải kèm thông tin quản lý):")
display(df_hw5)


#### Bài tập 6: Phân loại dải mô-men xoắn và đánh giá rủi ro hỏng hóc (CASE WHEN & GROUP BY)
**Bài toán:** Phân loại các mức mô-men xoắn hoạt động để đánh giá xem mô-men xoắn quá cao có làm tăng đột biến tỷ lệ hỏng hóc của máy móc hay không.

**Yêu cầu:**
Sử dụng biểu thức `CASE WHEN` để phân loại cột `torque_nm` thành cột mới có tên `torque_group`:
* `torque_nm < 30.0` ➔ `'Mô-men xoắn thấp (<30Nm)'`
* `torque_nm` từ `30.0` đến `60.0` ➔ `'Mô-men xoắn tiêu chuẩn (30-60Nm)'`
* `torque_nm > 60.0` ➔ `'Mô-men xoắn cực cao (>60Nm)'`

Gom nhóm theo biểu thức `CASE WHEN` này để tính:
1. Tên nhóm: `torque_group`.
2. Tổng số máy trong từng nhóm (`COUNT(*)`), đặt alias là `total_machines`.
3. Tổng số máy bị hỏng (`SUM(machine_failure)`), đặt alias là `failed_machines`.
4. Tỷ lệ phần trăm máy bị hỏng (`ROUND((AVG(machine_failure) * 100)::numeric, 2)`), đặt alias là `failure_rate_pct`.
5. Sắp xếp (`ORDER BY`): Theo tổng số máy `total_machines` giảm dần (`DESC`).


In [ ]:
# --- BÀI TẬP 6: Viết câu truy vấn SQL của bạn tại đây ---
query_hw6 = """
SELECT 
    CASE 
        WHEN torque_nm < 30.0 THEN 'Mô-men xoắn thấp (<30Nm)'
        WHEN torque_nm <= 60.0 THEN 'Mô-men xoắn tiêu chuẩn (30-60Nm)'
        ELSE 'Mô-men xoắn cực cao (>60Nm)'
    END AS torque_group,
    -- Điền các hàm tính toán COUNT, SUM, ROUND(AVG...)
FROM ai4i2020
GROUP BY 
    CASE 
        WHEN torque_nm < 30.0 THEN 'Mô-men xoắn thấp (<30Nm)'
        WHEN torque_nm <= 60.0 THEN 'Mô-men xoắn tiêu chuẩn (30-60Nm)'
        ELSE 'Mô-men xoắn cực cao (>60Nm)'
    END
ORDER BY total_machines DESC;
"""

df_hw6 = pd.read_sql(query_hw6, con=engine)
print("Kết quả Bài 6 (Thống kê tỷ lệ sự cố theo dải mô-men xoắn):")
display(df_hw6)
